# ETP Primitives Deep Dive

## Introduction

ETP (Eligibility Trace Propagation) primitives are JAX custom primitives that **mark weight operations** in the computational graph. They replace the old `ETraceOp` / JIT-name-matching system with a cleaner, more robust approach.

Key design principles:

- **Type identity, not string matching.** The compiler identifies ETP primitives by checking `eqn.primitive in ETP_PRIMITIVES` -- a set membership test on the primitive object itself. This is more reliable than the old approach of matching JIT function names.

- **All JAX rules are auto-derived.** Each primitive's `impl` delegates to standard JAX ops (e.g., `x @ w`, `jax.lax.conv_general_dilated`). The `register_primitive()` helper automatically derives JIT, grad, vmap, JVP, transpose, abstract_eval, and lowering rules from the implementation. No hand-written derivative formulas are needed.

- **Only ETP-specific rules need hand-writing.** Two (optionally three) rule registries capture the online-learning-specific logic:
  - `etp_rules_yw_to_w` -- D-RTRL trace propagation
  - `etp_rules_xy_to_dw` -- weight gradient computation
  - `etp_rules_init_state` -- trace state initialization

- **Primitive-based parameter selection.** A parameter participates in ETP if and only if it flows through an ETP primitive (`braintrace.matmul`, `braintrace.element_wise`, etc.). Parameters used with regular JAX ops are automatically excluded -- no special parameter class is needed.

In [1]:
import jax
import jax.numpy as jnp

import braintrace

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


## The Five Primitive Functions

`braintrace` provides five user-facing ETP primitive functions:

| Function | Underlying primitives | Purpose |
|---|---|---|
| `braintrace.matmul` | `etp_mm_p` (batched) / `etp_mv_p` (unbatched) | Dense matrix multiplication |
| `braintrace.element_wise` | `etp_elemwise_p` | Element-wise (diagonal) weight ops |
| `braintrace.conv` | `etp_conv_p` | Convolution |
| `braintrace.sparse_matmul` | `etp_sp_mm_p` / `etp_sp_mv_p` | Sparse matrix multiplication |
| `braintrace.lora_matmul` | `etp_lora_mm_p` / `etp_lora_mv_p` | LoRA (Low-Rank Adaptation) matmul |

Each function auto-dispatches between batched and unbatched variants based on input dimensionality.

### 1. `braintrace.matmul(x, weight, bias=None)` -- Dense Matrix Multiplication

Computes $y = x \, @ \, w \; (+ b)$.

Auto-dispatches based on `x.ndim`:
- `x.ndim >= 2` --> `etp_mm_p` (batched): expects `x` of shape `(batch, in_features)`
- `x.ndim == 1` --> `etp_mv_p` (unbatched): expects `x` of shape `(in_features,)`

In [2]:
# Batched matmul: x has shape (batch, in_features)
x_batched = jnp.ones((4, 3))    # batch=4, in_features=3
w = jnp.ones((3, 5))            # in_features=3, out_features=5

y_batched = braintrace.matmul(x_batched, w)
print("Batched output shape:", y_batched.shape)   # (4, 5)

# Unbatched matmul: x has shape (in_features,)
x_single = jnp.ones((3,))       # in_features=3

y_single = braintrace.matmul(x_single, w)
print("Unbatched output shape:", y_single.shape)   # (5,)

Batched output shape: (4, 5)
Unbatched output shape: (5,)


In [3]:
# With bias
b = jnp.zeros((5,))

y_with_bias = braintrace.matmul(x_batched, w, bias=b)
print("With bias:", y_with_bias.shape)              # (4, 5)

With bias: (4, 5)


### 2. `braintrace.element_wise(weight, fn=None)` -- Element-wise Operation

Applies an optional function `fn` to the weight and passes the result through a marker primitive. The operation is treated as *diagonal* in the hidden-state space.

$$y = \texttt{fn}(w)$$

This is commonly used for:
- Gating mechanisms in RNNs (learnable gate biases)
- Learnable time constants or thresholds in spiking neural networks
- Any parameter that enters the computation element-wise

In [4]:
# Identity (default fn): just marks the weight for ETP
w_gate = jnp.array([0.5, -0.3, 0.8, 0.1])

y_identity = braintrace.element_wise(w_gate)
print("Identity:", y_identity)

# With a transformation function
y_sigmoid = braintrace.element_wise(w_gate, fn=jax.nn.sigmoid)
print("Sigmoid:", y_sigmoid)

# With absolute value (e.g., enforcing positive time constants)
y_abs = braintrace.element_wise(w_gate, fn=jnp.abs)
print("Abs:", y_abs)

Identity: [ 0.5 -0.3  0.8  0.1]
Sigmoid: [0.62245935 0.4255575  0.6899745  0.5249792 ]


Abs: [0.5 0.3 0.8 0.1]


### 3. `braintrace.conv(x, kernel, bias=None, *, strides, padding, ...)` -- Convolution

ETP-aware convolution that wraps `jax.lax.conv_general_dilated`. Computes:

$$y = \text{conv}(x, \text{kernel}) \; (+ b)$$

**Important**: Always expects a batch dimension on `x`.

Supports all parameters of `jax.lax.conv_general_dilated`: `strides`, `padding`, `lhs_dilation`, `rhs_dilation`, `feature_group_count`, `batch_group_count`, and `dimension_numbers`.

In [5]:
# 1D convolution example
# x: (batch, spatial, channels) with dimension_numbers
x_1d = jnp.ones((2, 16, 3))         # batch=2, length=16, in_channels=3
kernel_1d = jnp.ones((4, 3, 8))     # kernel_size=4, in_channels=3, out_channels=8

y_conv = braintrace.conv(
    x_1d, kernel_1d,
    strides=(1,),
    padding='SAME',
    dimension_numbers=('NWC', 'WIO', 'NWC'),
)
print("Conv1D output shape:", y_conv.shape)  # (2, 16, 8)

Conv1D output shape: (2, 16, 8)


In [6]:
# 2D convolution example
x_2d = jnp.ones((2, 32, 32, 3))          # batch=2, H=32, W=32, in_channels=3
kernel_2d = jnp.ones((3, 3, 3, 16))      # kH=3, kW=3, in_channels=3, out_channels=16

y_conv2d = braintrace.conv(
    x_2d, kernel_2d,
    strides=(1, 1),
    padding='SAME',
    dimension_numbers=('NHWC', 'HWIO', 'NHWC'),
)
print("Conv2D output shape:", y_conv2d.shape)  # (2, 32, 32, 16)

Conv2D output shape: (2, 32, 32, 16)


### 4. `braintrace.sparse_matmul(x, weight_data, *, sparse_mat, bias=None)` -- Sparse Matmul

ETP-aware sparse matrix multiplication. Computes:

$$y = x \, @ \, \text{sparse}(w) \; (+ b)$$

The `sparse_mat` argument provides the sparse structure (indices), while `weight_data` contains only the non-zero values. This is useful for models with sparse connectivity patterns, such as biologically plausible neural networks or graph neural networks.

In [7]:
import saiunit as u
from saiunit import sparse as ss

# Create a sparse connectivity matrix
dense_w = jnp.where(
    jax.random.uniform(jax.random.PRNGKey(0), (50, 50)) < 0.1,
    jax.random.normal(jax.random.PRNGKey(1), (50, 50)),
    0.0
)
sparse_mat = ss.CSR.fromdense(dense_w)

# The learnable parameter is just the non-zero data
weight_data = sparse_mat.data

x_sp = jnp.ones((4, 50))  # batch=4, features=50
y_sp = braintrace.sparse_matmul(x_sp, weight_data, sparse_mat=sparse_mat)
print("Sparse matmul output shape:", y_sp.shape)  # (4, 50)

Sparse matmul output shape: (4, 50)


### 5. `braintrace.lora_matmul(x, B, A, *, alpha=1.0, bias=None)` -- LoRA Matmul

Low-Rank Adaptation matmul. Computes:

$$y = \alpha \cdot x \, @ \, B \, @ \, A \; (+ b)$$

where $B \in \mathbb{R}^{\text{in} \times \text{rank}}$ and $A \in \mathbb{R}^{\text{rank} \times \text{out}}$ are low-rank factors. This is useful for parameter-efficient fine-tuning of large models, where only the low-rank factors are trained.

In [8]:
in_features, out_features, rank = 64, 32, 4

B = jax.random.normal(jax.random.PRNGKey(0), (in_features, rank)) * 0.01
A = jax.random.normal(jax.random.PRNGKey(1), (rank, out_features)) * 0.01

x_lora = jnp.ones((8, in_features))  # batch=8

y_lora = braintrace.lora_matmul(x_lora, B, A, alpha=2.0)
print("LoRA output shape:", y_lora.shape)  # (8, 32)
print("LoRA output (first sample):", y_lora[0])

LoRA output shape: (8, 32)
LoRA output (first sample): [ 1.2993842e-04  2.9886682e-03 -5.7171844e-04  1.7000248e-03
  2.8961061e-03 -1.4734608e-03  1.5826351e-03  1.1692893e-04
 -5.4561032e-04 -1.3646147e-03  1.0654312e-04  2.0160272e-03
  3.1371990e-03  1.8710974e-03  2.9124888e-03 -8.2047645e-04
  9.7736251e-04 -1.1875220e-03 -2.1796541e-03 -5.8191392e-05
 -1.5415263e-03 -1.3381819e-03 -2.1044153e-03 -8.4472442e-04
  1.6430757e-05 -2.9564608e-04  4.5550600e-04 -1.6011632e-03
 -1.4516843e-03 -1.3209208e-03  3.2850087e-04 -6.0276774e-04]


## JAX Compatibility

All ETP primitives are fully compatible with JAX transformations. Since `register_primitive()` auto-derives JIT, grad, vmap, and JVP rules from the implementation, they work seamlessly with the standard JAX API.

In [9]:
x = jnp.ones((4, 3))
w = jnp.ones((3, 5))

# ---- JIT compilation ----
y_jit = jax.jit(braintrace.matmul)(x, w)
print("JIT output shape:", y_jit.shape)

JIT output shape: (4, 5)


In [10]:
# ---- Gradient computation ----
grad_fn = jax.grad(lambda w: jnp.sum(braintrace.matmul(x, w)))
dw = grad_fn(w)
print("Gradient shape:", dw.shape)
print("Gradient values:\n", dw)

Gradient shape: (3, 5)
Gradient values:
 [[4. 4. 4. 4. 4.]
 [4. 4. 4. 4. 4.]
 [4. 4. 4. 4. 4.]]


In [11]:
# ---- Vectorized mapping (vmap) ----
# vmap over a batch of inputs, each of shape (4, 3)
xs = jnp.ones((8, 4, 3))  # 8 different batches
vmap_fn = jax.vmap(lambda x_i: braintrace.matmul(x_i, w))
ys = vmap_fn(xs)
print("vmap output shape:", ys.shape)  # (8, 4, 5)

vmap output shape: (8, 4, 5)


In [12]:
# ---- JVP (forward-mode differentiation) ----
primals = (x, w)
tangents = (jnp.ones_like(x), jnp.ones_like(w))

y_primal, y_tangent = jax.jvp(braintrace.matmul, primals, tangents)
print("JVP primal shape:", y_primal.shape)
print("JVP tangent shape:", y_tangent.shape)

JVP primal shape: (4, 5)
JVP tangent shape: (4, 5)


In [13]:
# ---- Composability: JIT + grad + vmap ----
@jax.jit
def batched_grad(xs, w):
    """Compute per-sample gradients w.r.t. the weight."""
    def single_grad(x_i):
        return jax.grad(lambda w_: jnp.sum(braintrace.matmul(x_i, w_)))(w)
    return jax.vmap(single_grad)(xs)

xs = jnp.ones((8, 4, 3))
per_sample_grads = batched_grad(xs, w)
print("Per-sample gradients shape:", per_sample_grads.shape)  # (8, 3, 5)

Per-sample gradients shape: (8, 3, 5)


## Argument Conventions

Each ETP primitive follows specific conventions for its input variables (`invars`) and static parameters. Understanding these conventions is essential when working with the compiler or adding custom primitives.

| Primitive | `invars[0]` | `invars[1]` | `invars[2]` | `invars[3]` | Static params |
|---|---|---|---|---|---|
| `etp_mm_p` / `etp_mv_p` | input `x` | weight `w` | bias `b` (opt) | -- | `has_bias` |
| `etp_elemwise_p` | processed `y` | -- | -- | -- | (none) |
| `etp_conv_p` | input `x` | kernel `w` | bias `b` (opt) | -- | `has_bias`, `strides`, `padding`, ... |
| `etp_sp_mm_p` / `etp_sp_mv_p` | input `x` | weight data | bias `b` (opt) | -- | `sparse_mat`, `has_bias` |
| `etp_lora_mm_p` / `etp_lora_mv_p` | input `x` | matrix `B` | matrix `A` | bias `b` (opt) | `alpha`, `has_bias` |

Notes:
- The weight variable is always at index 1 for matmul/conv/sparse/lora, and index 0 for elemwise.
- The `has_bias` flag is a static parameter (not a traced value) that controls whether the optional bias argument is present.
- For convolution, all `jax.lax.conv_general_dilated` parameters are passed as static params.

## Rule Registries

ETP uses **four** global dictionaries to store operation-specific rules. These are the *only* things that need hand-writing -- all standard JAX rules are auto-derived from the implementation function.

### `ETP_RULES_YW_TO_W` -- D-RTRL Trace Propagation

Signature: `(hidden_dim, trace, **params) -> updated_trace`

Defines how the eligibility trace is propagated through the operation.
Called during the D-RTRL trace update step.

### `ETP_RULES_XY_TO_DW` -- Weight Gradient Computation

Signature: `(x, hidden_dim, weight, **params) -> dw`

Computes the weight gradient (or immediate influence term). Used by both
`D_RTRL` and `ES_D_RTRL` when reading the trace into a parameter update.

### `ETP_RULES_INIT_DRTRL` -- D-RTRL Trace Initializer

Signature: `(x_var, y_var, weight, num_hidden_state) -> zeros`

Returns a zero-filled array (or pytree of arrays) shaped to hold the
**parameter-dimension** trace used by `D_RTRL` / `ParamDimVjpAlgorithm`.

### `ETP_RULES_INIT_PP` -- pp_prop / ES-D-RTRL Trace Initializer

Signature: `(x_var, y_var, weight, num_hidden_state) -> zeros`

Returns a zero-filled array shaped to hold the **input/output-dimension**
trace used by `ES_D_RTRL` / `pp_prop` / `IODimVjpAlgorithm`.

The two `INIT_*` registries exist because the two algorithm families
factorise the trace differently. Both are required for a primitive that
should support both algorithms.

In [14]:
from braintrace._etrace_op import (
    ETP_RULES_YW_TO_W,
    ETP_RULES_XY_TO_DW,
    ETP_RULES_INIT_DRTRL,
    ETP_RULES_INIT_PP,
    ETP_PRIMITIVES,
    BATCHED_PRIMITIVES,
)

print("All ETP primitives:")
for p in sorted(ETP_PRIMITIVES, key=lambda p: p.name):
    batched_tag = " [batched]" if p in BATCHED_PRIMITIVES else ""
    print(f"  {p.name}{batched_tag}")

print("\nTrace propagation rules (ETP_RULES_YW_TO_W):")
for p in sorted(ETP_RULES_YW_TO_W.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

print("\nWeight gradient rules (ETP_RULES_XY_TO_DW):")
for p in sorted(ETP_RULES_XY_TO_DW.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

print("\nD-RTRL init rules (ETP_RULES_INIT_DRTRL):")
for p in sorted(ETP_RULES_INIT_DRTRL.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

print("\npp_prop init rules (ETP_RULES_INIT_PP):")
for p in sorted(ETP_RULES_INIT_PP.keys(), key=lambda p: p.name):
    print(f"  {p.name}")

All ETP primitives:
  etp_conv [batched]
  etp_elemwise
  etp_lora_mm [batched]
  etp_lora_mv
  etp_mm [batched]
  etp_mv
  etp_sp_mm [batched]
  etp_sp_mv

Trace propagation rules (ETP_RULES_YW_TO_W):
  etp_conv
  etp_elemwise
  etp_lora_mm
  etp_lora_mv
  etp_mm
  etp_mv
  etp_sp_mm
  etp_sp_mv

Weight gradient rules (ETP_RULES_XY_TO_DW):
  etp_conv
  etp_elemwise
  etp_lora_mm
  etp_lora_mv
  etp_mm
  etp_mv
  etp_sp_mm
  etp_sp_mv

D-RTRL init rules (ETP_RULES_INIT_DRTRL):
  etp_conv
  etp_elemwise
  etp_lora_mm
  etp_lora_mv
  etp_mm
  etp_mv
  etp_sp_mm
  etp_sp_mv

pp_prop init rules (ETP_RULES_INIT_PP):
  etp_conv
  etp_elemwise
  etp_lora_mm
  etp_lora_mv
  etp_mm
  etp_mv
  etp_sp_mm
  etp_sp_mv


## Adding a Custom Primitive

Adding a new ETP primitive takes only a few steps. Here we create a scaled matrix multiplication primitive as an example:

$$y = \text{scale} \cdot (x \, @ \, w)$$

In [15]:
import braintrace
from braintrace import register_primitive


# Step 1: Define the implementation
# This is a plain JAX function -- no special annotations needed.
def _scaled_matmul_impl(x, w, *, scale=1.0, has_bias=False):
    return scale * (x @ w)


# Step 2: Register as an ETP primitive
# register_primitive() returns an ``ETPPrimitive`` and auto-derives
# all standard JAX rules (jit, grad, vmap, jvp) from the implementation.
scaled_mm_p = register_primitive('etp_scaled_mm', _scaled_matmul_impl, batched=True)

print("Primitive registered:", scaled_mm_p)
print("Type:", type(scaled_mm_p).__name__)

Primitive registered: etp_scaled_mm
Type: ETPPrimitive


In [16]:
# Step 3: Register the four ETP-specific rules.
# ``ETPPrimitive`` exposes a ``register_*`` method per registry so you
# do not need to mutate the global dicts by hand.

# Trace propagation: how the eligibility trace updates through this op.
scaled_mm_p.register_yw_to_w(
    lambda hidden_dim, trace, **p: trace * jnp.expand_dims(hidden_dim, axis=1)
)

# Weight gradient: immediate influence of weight on output.
scaled_mm_p.register_xy_to_dw(
    lambda x, hidden_dim, w, **p:
        jax.vjp(lambda w_: p.get('scale', 1.0) * (x @ w_), w)[1](hidden_dim)[0]
)

# D-RTRL trace initialiser: parameter-dimension trace storage.
scaled_mm_p.register_init_drtrl(
    lambda x_var, y_var, weight, n:
        jnp.zeros((x_var.aval.shape[0], *jnp.shape(weight.value), n))
)

# pp_prop trace initialiser: IO-dimension trace storage.
scaled_mm_p.register_init_pp(
    lambda x_var, y_var, weight, n:
        jnp.zeros((*y_var.aval.shape, n), dtype=y_var.aval.dtype)
)

# Equivalent shorthand: ``scaled_mm_p.register_etp_rules(yw_to_w=..., xy_to_dw=..., init_drtrl=..., init_pp=...)``.

print("yw_to_w registered:", scaled_mm_p in ETP_RULES_YW_TO_W)
print("xy_to_dw registered:", scaled_mm_p in ETP_RULES_XY_TO_DW)
print("init_drtrl registered:", scaled_mm_p in ETP_RULES_INIT_DRTRL)
print("init_pp registered:", scaled_mm_p in ETP_RULES_INIT_PP)

yw_to_w registered: True
xy_to_dw registered: True
init_drtrl registered: True
init_pp registered: True


In [17]:
# Step 4: Use the custom primitive

x = jnp.ones((4, 3))
w = jnp.ones((3, 5))

# Direct usage via primitive.bind()
y = scaled_mm_p.bind(x, w, scale=2.0, has_bias=False)
print("Output shape:", y.shape)
print("Output (scale=2.0):\n", y)

# Verify: should equal 2.0 * (x @ w)
y_expected = 2.0 * (x @ w)
print("\nMatches expected:", jnp.allclose(y, y_expected))

Output shape: (4, 5)
Output (scale=2.0):
 [[6. 6. 6. 6. 6.]
 [6. 6. 6. 6. 6.]
 [6. 6. 6. 6. 6.]
 [6. 6. 6. 6. 6.]]



Matches expected: True


In [18]:
# The custom primitive works with all JAX transformations automatically

# JIT
y_jit = jax.jit(lambda x, w: scaled_mm_p.bind(x, w, scale=2.0, has_bias=False))(x, w)
print("JIT works:", jnp.allclose(y_jit, y_expected))

# Grad
dw = jax.grad(lambda w: jnp.sum(scaled_mm_p.bind(x, w, scale=2.0, has_bias=False)))(w)
print("Grad shape:", dw.shape)

# Vmap
xs = jnp.ones((8, 4, 3))
ys = jax.vmap(lambda xi: scaled_mm_p.bind(xi, w, scale=2.0, has_bias=False))(xs)
print("Vmap output shape:", ys.shape)

JIT works: True


Grad shape: (3, 5)


Vmap output shape: (8, 4, 5)


## Spec-based Registration

For testing or tooling that wants a single object describing every aspect of a primitive, use the **spec** form. An :class:`ETPPrimitiveSpec` is a dataclass-like value bundling the implementation, the four rules, and the invar/outvar layout the compiler should use. Pass it to :func:`register_primitive_spec` and you get back a fully-wired :class:`ETPPrimitive`. The spec is stored alongside the primitive so the compiler can query it with :func:`get_primitive_spec`.

The spec form is equivalent to the class-based form -- pick whichever style fits your codebase.

In [19]:
import braintrace


def _spec_impl(x, w, *, scale=1.0, has_bias=False):
    return scale * (x @ w)


spec = braintrace.ETPPrimitiveSpec(
    name='etp_spec_demo',
    impl=_spec_impl,
    yw_to_w=lambda hidden_dim, trace, **p:
        trace * jnp.expand_dims(hidden_dim, axis=1),
    xy_to_dw=lambda x, hidden_dim, w, **p:
        jax.vjp(lambda w_: p.get('scale', 1.0) * (x @ w_), w)[1](hidden_dim)[0],
    init_drtrl=lambda x_var, y_var, w, n:
        jnp.zeros((x_var.aval.shape[0], *jnp.shape(w.value), n)),
    init_pp=lambda x_var, y_var, w, n:
        jnp.zeros((*y_var.aval.shape, n), dtype=y_var.aval.dtype),
    weight_invar_index=1,   # ``invars[1]`` is the weight (matches mm/mv/conv)
    x_invar_index=0,        # ``invars[0]`` is the input
    batched=True,
)

spec_p = braintrace.register_primitive_spec(spec)

# The spec is recoverable from the primitive at any time.
assert braintrace.get_primitive_spec(spec_p) is spec
print("Primitive registered from spec:", spec_p)

Primitive registered from spec: etp_spec_demo


## The `gradient_enabled` Flag

`register_primitive()` accepts a `gradient_enabled` keyword (default `False`). It controls how the compiler treats this primitive when walking from a weight's output back to a hidden state.

| `gradient_enabled` | Compiler behaviour | Example |
|---|---|---|
| `False` (default) | Treats the primitive as a **tail boundary**. A preceding ETP weight whose only path to a hidden state passes through this primitive is **excluded** from ETP, because per-primitive ETP rules cannot express weight-then-weight composition. | All trainable matmul/conv/sparse/LoRA primitives use this. |
| `True` | The primitive is **identity-like** and may sit on the tail of the `y -> h` walk. Its presence does not exclude an upstream ETP weight. | Only `etp_elemwise_p` -- intended for gating biases, learnable thresholds, etc. |

Use `gradient_enabled=True` only when the primitive's `xy_to_dw` rule is itself an identity-like passthrough; mark all genuinely *trainable* ops with the default. The "weight -> weight -> hidden" exclusion is what makes per-primitive ETP rules sound -- see ``advanced/limitations.ipynb`` for a worked example with ``GRUCell`` (3 Linears, only 2 ETP relations).

## Summary

ETP primitives provide a clean, extensible foundation for online learning in recurrent networks:

- **8 built-in primitives** cover the most common use cases: dense matmul (mm/mv), element-wise ops, convolution, sparse matmul (mm/mv), and LoRA matmul (mm/mv).

- **Custom primitives can be added in approximately 10 lines of code**: define an implementation function, call `register_primitive()`, and register the four ETP-specific rules via `register_yw_to_w / register_xy_to_dw / register_init_drtrl / register_init_pp` (or pass them all at once with `register_etp_rules(...)` or via `ETPPrimitiveSpec` + `register_primitive_spec`).

- **All JAX transformations (JIT, grad, vmap, JVP) work automatically** -- only the four online-learning-specific rules need hand-writing.

- **Parameter selection is primitive-based** -- every `brainstate.ParamState` is eligible for ETP, and participation depends only on whether a `braintrace.*` ETP primitive consumed it. Use `gradient_enabled=True` exclusively for identity-like ops such as `etp_elemwise_p`.